# 09  IPO trade mark signals (a brand and growth signal)

NB08 added property ownership. This notebook adds a fourth signal: **which of our companies hold a
UK trade mark.** Registering a trade mark is a sign a firm is investing in a brand, which usually
goes with growth, marketing and a longer-term plan. The data is HM Intellectual Property Office
(IPO) open trade mark data, free under the Open Government Licence.

Honest note on quality: this source is weaker than the last one. Unlike Land Registry, the IPO file
has **no company registration number**, so we can only match by name. It also gives only the
**postcode area** of the owner (for example `SL3`, not the full `SL3 9EH`). So we confirm a name
match by postcode area, which is coarser than the full-postcode check we used before. That is why
the confidence scores here are lower, and why this signal is best treated as an enrichment layer
rather than a precise one.

## Getting the IPO file (one-time, no key needed)
There is no API for this data, so download it once:
1. Go to https://www.gov.uk/government/publications/ipo-trade-mark-data-release
2. Download **Open Data: Domestic UK Applications** (a ZIP, about 60 MB, tab separated text).
3. Drop that ZIP (or the tab separated file inside it) into your `MyDrive/Lloyds` folder.

The notebook finds it, unzips if needed, and reads it. If you prefer, paste the direct download link
into `TM_URL` in Section 2 and it will fetch the file for you. Please do not commit the file to
GitHub; like the other data it stays out of the repo.

## How to run
Run NB05 first so `lloyds.duckdb` exists, put the IPO file in `MyDrive/Lloyds`, then run this
notebook top to bottom. It updates the same database.

## 1. Install and import

Same as the other notebooks: makes sure DuckDB and RapidFuzz are available, downloading them if they are not already there.

In [ ]:
import sys, subprocess
for pkg in ["duckdb", "rapidfuzz"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

Switches the tools on. RapidFuzz compares company names that are close but not identical.

In [ ]:
import duckdb
import pandas as pd
import requests
import re, io, zipfile, csv
from collections import defaultdict
from datetime import datetime, timezone
from rapidfuzz import process, fuzz

print("duckdb", duckdb.__version__, "| pandas", pd.__version__)

## 2. Locate the database and the IPO file
Find the `lloyds.duckdb` that NB05 built, and find the IPO trade mark file you downloaded. If you
put the ZIP in the folder it is unzipped automatically. The database must already exist (run NB05
first).

This box finds the database and the IPO file. If you set `TM_URL` to the direct download link it fetches the file for you; otherwise it looks for a ZIP or tab separated file you have put in the folder. `TM_MAX_ROWS` is a dial (None = whole file). `REGISTERED_ONLY` keeps only live, registered marks, which is the cleanest brand signal.

In [ ]:
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/Lloyds")     # same folder as NB05
    DB_PATH = WORK_DIR / "lloyds.duckdb"
else:
    WORK_DIR = Path("..").resolve() / "data" / "processed"
    DB_PATH = WORK_DIR / "lloyds.duckdb"

assert DB_PATH.exists(), f"lloyds.duckdb not found at {DB_PATH}. Run NB05 first."
print("DB:", DB_PATH)

TM_URL = ""              # optional: paste the direct .zip download link from the IPO page
TM_MAX_ROWS = None       # None = whole file; set e.g. 200000 for a quick trial
REGISTERED_ONLY = True   # keep only live, registered marks
NOW = datetime.now(timezone.utc).isoformat(timespec="seconds")

def _extract_zip_to_dir(zip_bytes_or_path, work_dir):
    zf = zipfile.ZipFile(zip_bytes_or_path)
    inner = [n for n in zf.namelist() if n.lower().endswith((".txt", ".tsv", ".tab", ".csv"))]
    if not inner:
        raise RuntimeError(f"No data file inside the zip. Contents: {zf.namelist()}")
    zf.extract(inner[0], work_dir)
    return work_dir / inner[0]

# 1) download if a URL was given; 2) else unzip a local zip; 3) else use a local text file
TM_PATH = None
if TM_URL:
    print("downloading IPO file ...")
    blob = requests.get(TM_URL, timeout=600).content
    TM_PATH = _extract_zip_to_dir(io.BytesIO(blob), WORK_DIR) if TM_URL.lower().endswith(".zip") \
        else (WORK_DIR / Path(TM_URL).name)
    if not TM_URL.lower().endswith(".zip"):
        TM_PATH.write_bytes(blob)

if TM_PATH is None:
    zips = [p for p in WORK_DIR.glob("*.zip")
            if any(k in p.name.lower() for k in ("tm", "trade", "domestic", "ipo"))]
    if zips:
        TM_PATH = _extract_zip_to_dir(sorted(zips)[-1], WORK_DIR)

if TM_PATH is None:
    texts = sorted(WORK_DIR.glob("*.txt")) + sorted(WORK_DIR.glob("*.tsv")) + sorted(WORK_DIR.glob("*.tab"))
    if not texts:
        raise FileNotFoundError(
            f"No IPO trade mark file found in {WORK_DIR}.\n"
            "Download 'Open Data: Domestic UK Applications' (a zip) from\n"
            "  https://www.gov.uk/government/publications/ipo-trade-mark-data-release\n"
            f"and drop it into {WORK_DIR} (or paste its link into TM_URL above).")
    TM_PATH = texts[-1]

print("IPO file:", TM_PATH.name)

## 3. Helpers
The same name cleaner as the other notebooks, plus small postcode helpers. The IPO gives only the postcode area, so we compare it against the area part of each company's full postcode.

This box recreates the name cleaner so companies match the same way everywhere, and adds postcode helpers. Because the IPO only gives a postcode area (like `SL3`), we reduce each company's full postcode to its area part and compare those.

In [ ]:
_SUFFIXES = [
    "LIMITED", "LTD", "PLC", "PUBLIC LIMITED COMPANY", "LLP",
    "LIMITED LIABILITY PARTNERSHIP", "LP", "CIC", "CIO",
    "COMPANY", "CO", "AND", "THE",
]
_SUFFIX_RE = re.compile(r"\b(" + "|".join(_SUFFIXES) + r")\b")

def normalise_name(name):
    if name is None:
        return None
    s = str(name).upper()
    s = re.sub(r"[^A-Z0-9 ]", " ", s)
    s = _SUFFIX_RE.sub(" ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s or None

def normalise_postcode(pc):
    if pc is None:
        return None
    s = re.sub(r"\s+", "", str(pc).upper())
    return s or None

def outward_code(full_pc):
    # reduce a full postcode to its outward part: the UK inward part is always the last 3
    # characters (one digit + two letters), so the outward part is everything before that.
    s = normalise_postcode(full_pc)
    if not s:
        return None
    return s[:-3] if len(s) > 3 else s

def area_match(ipo_area, company_outward):
    # the IPO area code (e.g. "SL" or "SL3") should be the start of the company's outward code
    if not ipo_area or not company_outward:
        return False
    return company_outward.startswith(ipo_area)

print("helpers ok")

## 4. Load the spine and build the match indexes
We pull the company number, normalised name, and postcode area for every company, then build an
exact-name index and first-word blocks so the fuzzy step stays fast.

This box reads the master company list and builds two lookups keyed on the simplified name, each carrying the company's postcode area so a name match can be confirmed by location.

In [ ]:
con = duckdb.connect(str(DB_PATH))
spine = con.execute("SELECT company_number, name_norm, postcode FROM companies").df()
print(f"spine: {len(spine):,} companies")

by_name = defaultdict(list)     # name_norm -> [(company_number, outward), ...]
blocks  = defaultdict(list)     # first word -> [(name_norm, company_number, outward), ...]
for cn, nm, pc in zip(spine["company_number"], spine["name_norm"], spine["postcode"]):
    if not nm:
        continue
    outw = outward_code(pc)
    by_name[nm].append((cn, outw))
    blocks[nm.split(" ")[0]].append((nm, cn, outw))
print(f"exact-name keys: {len(by_name):,} | first-word blocks: {len(blocks):,}")

## 5. The matching ladder
There is no company number here, so every match is by name, confirmed by postcode area where the
IPO gives one. Because a postcode area covers a whole town, this is a softer check than the
full-postcode confirmation used for contracts and property, so the confidence scores are lower.

Tiers, strongest first:
1. `name_exact_area` (0.8) - exact normalised name and the postcode area agrees.
2. `name_fuzzy_area` (~0.7) - RapidFuzz match above a high cutoff, postcode area also agrees.
3. `name_exact_unconfirmed` (0.45) - exact name, but no postcode area to confirm with.

This box defines the matching ladder. It tries an exact name with a matching postcode area first, then a close fuzzy name with a matching area. If the IPO gives an area that does not match, we do not claim a match, which guards against two firms sharing a name.

In [ ]:
FUZZY_CUTOFF = 93   # higher than the other notebooks because the area check is weaker

def match_owner(name, ipo_area):
    nm = normalise_name(name)
    if not nm:
        return None, 0.0, "no_match"
    area = normalise_postcode(ipo_area)

    # 1. exact normalised name, confirmed by postcode area where the record gives one
    if nm in by_name:
        cands = by_name[nm]
        confirmed = [c for c, outw in cands if area_match(area, outw)]
        if confirmed:
            return confirmed[0], 0.8, "name_exact_area"
        if area:
            return None, 0.0, "name_exact_area_mismatch"   # same name, different area
        if len(cands) == 1:
            return cands[0][0], 0.45, "name_exact_unconfirmed"
        return None, 0.0, "name_exact_ambiguous"

    # 2. fuzzy within the first-word block, only when the postcode area also agrees
    if area:
        bucket = blocks.get(nm.split(" ")[0])
        if bucket:
            choices = [b[0] for b in bucket]
            for _, score, idx in process.extract(
                    nm, choices, scorer=fuzz.WRatio, score_cutoff=FUZZY_CUTOFF, limit=5):
                if area_match(area, bucket[idx][2]):
                    return bucket[idx][1], round(score / 100 * 0.75, 3), "name_fuzzy_area"

    return None, 0.0, "no_match"

# sanity check: a made-up name should not match
print(match_owner("A COMPANY THAT DOES NOT EXIST ZZZ", "ZZ9"))

## 6. Read the IPO trade mark file and tidy it
The file is a tab separated dump, so we read it defensively: try a header row, fall back to the
known column layout if there is none, and skip the odd malformed line. We keep the owner name and
postcode area, the mark text, the status and the key dates. If `REGISTERED_ONLY` is on, we keep
only marks that are registered and not expired.

This box reads the IPO file. Trade mark dumps are messy, so it tries a few encodings, handles the file whether or not it has a header row, and skips broken lines. It then keeps the owner name, the postcode area, the brand text, the status and the dates, and optionally filters to live registered marks only.

In [ ]:
IPO_COLS = ["Trade mark", "Hyperlink", "Mark Text", "Name", "Postcode", "Region", "Country",
            "Status", "Category of Mark", "Mark Type", "Series", "No of marks in series",
            "Filed", "Published", "Registered", "Expired", "Renewal due date", "Class"]

def read_ipo(path, max_rows):
    # the IPO file is pipe separated and UTF-16 encoded (it starts with a byte-order mark), so try
    # UTF-16 first. It has a header row with real column names, plus extra Class1..ClassN columns
    # we simply ignore.
    df = None
    for enc in ("utf-16", "utf-8-sig", "latin-1"):
        try:
            df = pd.read_csv(path, sep="|", dtype=str, encoding=enc, engine="python",
                             quoting=csv.QUOTE_NONE, on_bad_lines="skip", nrows=max_rows)
            if df.shape[1] > 1:      # columns split properly with this encoding
                break
        except (UnicodeDecodeError, UnicodeError):
            continue
    if df is None or df.shape[1] <= 1:
        raise RuntimeError("Could not parse the IPO file as pipe separated UTF-16. "
                           f"First columns seen: {list(df.columns)[:5] if df is not None else 'none'}")
    df.columns = [c.strip() for c in df.columns]
    if "Name" not in df.columns:
        raise RuntimeError(f"No 'Name' column after parsing. Columns: {list(df.columns)[:20]}")
    return df

raw = read_ipo(TM_PATH, TM_MAX_ROWS)
print(f"IPO rows read: {len(raw):,}")

marks = pd.DataFrame({
    "owner_name":  raw.get("Name"),
    "ipo_area":    raw.get("Postcode"),
    "region":      raw.get("Region"),
    "mark_text":   raw.get("Mark Text"),
    "status":      raw.get("Status"),
    "filed":       raw.get("Filed"),
    "registered":  raw.get("Registered"),
    "expired":     raw.get("Expired"),
})
marks = marks[marks["owner_name"].notna()]

if REGISTERED_ONLY:
    before = len(marks)
    marks = marks[marks["registered"].notna() & marks["expired"].isna()]
    print(f"kept live registered marks: {len(marks):,} of {before:,}")

print(f"marks to match: {len(marks):,}")
print(f"  with a postcode area: {marks['ipo_area'].notna().sum():,}")
marks.head(3)

## 7. Match owners to the spine and write trademark signals
Run each mark's owner through the name ladder, keep the ones that match a company in our dataset,
and write them into `signals`. We clear any previous IPO rows first so re-runs do not double count.
A company can hold several marks, so it can carry several rows.

This box runs every trade mark owner through the name and postcode-area ladder, keeps the ones that match a company in our list, and saves them into the signals table as trademark events. It clears old IPO rows first so re-running does not double count.

In [ ]:
rows = []
for r in marks.itertuples(index=False):
    cn, conf, method = match_owner(r.owner_name, r.ipo_area)
    if cn:
        rows.append((cn, r.mark_text, r.registered, r.filed, conf, method))

matched = pd.DataFrame(rows, columns=["company_number", "mark_text", "registered", "filed",
                                      "confidence", "method"])
# signal_date: the registration date where present, otherwise the filing date
# (IPO dates are ISO YYYY-MM-DD)
reg = pd.to_datetime(matched["registered"], errors="coerce", format="%Y-%m-%d")
fil = pd.to_datetime(matched["filed"], errors="coerce", format="%Y-%m-%d")
matched["signal_date"] = reg.fillna(fil).dt.date
matched["value"] = pd.NA                       # a trade mark has no monetary value
matched["detail"] = matched["mark_text"].astype(str).str.slice(0, 200)
matched["signal_type"] = "trademark"
matched["source"] = "ipo_trademarks"
matched["retrieved_at"] = NOW

print(f"owners matched to a company in the dataset: {len(matched):,} of {len(marks):,}")
print("\nby method:")
print(matched["method"].value_counts().to_string())

con.execute("DELETE FROM signals WHERE source = 'ipo_trademarks'")
ins = matched[["company_number", "signal_type", "signal_date", "value", "detail",
               "source", "confidence", "retrieved_at"]]
con.register("tmp_sig", ins)
con.execute("""INSERT INTO signals
               SELECT company_number, signal_type, signal_date, value, detail,
                      source, confidence, retrieved_at
               FROM tmp_sig""")
con.unregister("tmp_sig")
print(f"\nwritten to signals: {len(ins):,} rows")

## 8. Summary and a look at the result
How many companies in the dataset now hold a trade mark, and a sample of the biggest holders.

This box counts how many companies now carry a trademark signal and lists the ones holding the most marks.

In [ ]:
n_companies = con.execute(
    "SELECT count(DISTINCT company_number) FROM signals WHERE source='ipo_trademarks'"
).fetchone()[0]
total = con.execute("SELECT count(*) FROM companies").fetchone()[0]
print(f"companies holding a trade mark: {n_companies:,} of {total:,} ({n_companies/total:.2%})")

sample = con.execute("""
    SELECT c.company_name, c.sector, count(*) AS trademarks, round(avg(s.confidence), 2) AS avg_conf
    FROM signals s JOIN companies c ON c.company_number = s.company_number
    WHERE s.source = 'ipo_trademarks'
    GROUP BY c.company_name, c.sector
    ORDER BY trademarks DESC
    LIMIT 10
""").df()
sample

## 9. Visualise the matching and the signals
Four pictures: how the marks narrow down to a confident match, which method did the matching, which companies hold the most marks, and how the matched marks spread across the country.

This box draws four charts: the funnel from marks down to matches, the matches by method, the top companies by number of trade marks, and the matched marks by region.

In [ ]:
import matplotlib.pyplot as plt

if len(matched) == 0:
    print("no matches to visualise yet. Check the IPO file loaded and NB05 built the full spine.")
else:
    fig, ax = plt.subplots(2, 2, figsize=(13, 9))

    # A. funnel: from marks down to a confident match
    funnel = {
        "marks\nto match": len(marks),
        "has a\npostcode area": int(marks["ipo_area"].notna().sum()),
        "matched to\nour dataset": len(matched),
    }
    ax[0, 0].bar(list(funnel.keys()), list(funnel.values()), color="#4477aa")
    ax[0, 0].set_title("From trade marks to companies matched")
    for i, v in enumerate(funnel.values()):
        ax[0, 0].text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)

    # B. matches by method (each maps to a confidence tier)
    mm = matched["method"].value_counts()
    ax[0, 1].barh(list(mm.index[::-1]), list(mm.values[::-1]), color="#228833")
    ax[0, 1].set_title("Matches by method (confidence tier)")
    for i, v in enumerate(mm.values[::-1]):
        ax[0, 1].text(v, i, f" {v:,}", va="center", fontsize=9)

    # C. top companies by number of trade marks held
    top = con.execute("""
        SELECT c.company_name, count(*) AS n
        FROM signals s JOIN companies c ON c.company_number = s.company_number
        WHERE s.source='ipo_trademarks'
        GROUP BY c.company_name ORDER BY n DESC LIMIT 10
    """).df()
    if len(top):
        ax[1, 0].barh(top["company_name"][::-1], top["n"][::-1], color="#ccbb44")
        ax[1, 0].set_title("Top companies by number of trade marks")
        ax[1, 0].set_xlabel("trade marks held")
        ax[1, 0].tick_params(axis="y", labelsize=8)

    # D. matched trade marks by year of registration
    bym = con.execute("""
        SELECT date_trunc('year', signal_date) AS y, count(*) AS n
        FROM signals WHERE source='ipo_trademarks' AND signal_date IS NOT NULL
        GROUP BY y ORDER BY y
    """).df()
    if len(bym):
        ax[1, 1].bar([str(x)[:4] for x in bym["y"]], bym["n"], color="#ee6677")
        ax[1, 1].set_title("Matched trade marks by year")
        ax[1, 1].tick_params(axis="x", rotation=90, labelsize=7)

    fig.suptitle("NB09 IPO trade marks: matching and the signals produced", fontsize=13)
    fig.tight_layout()
    plt.show()

The funnel shows the marks narrowing down to the ones we matched. Because there is no company number and only a postcode area, expect fewer and softer matches than the property source. The other charts show who holds the most marks and when they were registered.

This box saves and closes the database so the new signals are kept.

In [ ]:
con.close()
print("saved:", DB_PATH)

## Notes and what comes next
- This is a name-matched source with no company number, confirmed only by postcode area, so it is
  the softest of the four signals. Filter on `confidence >= 0.8` to keep only the exact-name,
  area-confirmed matches; the `name_exact_unconfirmed` rows (0.45) are the ones to treat with care.
- `REGISTERED_ONLY` keeps live registered marks, which is the cleanest brand signal. Turn it off to
  include pending applications as well.
- The signals table now holds four sources: contract_win, hiring, owns_property and trademark. That
  is a good spread of commercial signals to feed the combined dataset and the model.
- Naming note for the team: the news work uses notebook numbers 08-09 on another branch, so renumber
  if needed before merging to main.